# Notebook 1 — Exploratory Data Analysis
**Purpose:** Understand the BlueStock Nifty 100 financial dataset before building dashboards.
Produce at least 20 visualizations covering distributions, correlations, and sector aggregations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load data from SQLite (change to PostgreSQL connection string for production)
import sqlite3, os, sys
sys.path.insert(0, os.path.abspath('..'))

DB_PATH = os.path.join('..', 'db.sqlite3')
conn = sqlite3.connect(DB_PATH)

companies = pd.read_sql('SELECT * FROM dim_company', conn)
sectors = pd.read_sql('SELECT * FROM dim_sector', conn)
years = pd.read_sql('SELECT * FROM dim_year ORDER BY sort_order', conn)
health_labels = pd.read_sql('SELECT * FROM dim_health_label', conn)
pl = pd.read_sql('SELECT * FROM fact_profit_loss', conn)
bs = pd.read_sql('SELECT * FROM fact_balance_sheet', conn)
cf = pd.read_sql('SELECT * FROM fact_cash_flow', conn)
analysis = pd.read_sql('SELECT * FROM fact_analysis', conn)
ml_scores = pd.read_sql('SELECT * FROM fact_ml_scores', conn)
pros_cons = pd.read_sql('SELECT * FROM fact_pros_cons', conn)

# Merge year labels and sector names
pl = pl.merge(years[['year_id', 'year_label', 'sort_order']], on='year_id')
bs = bs.merge(years[['year_id', 'year_label', 'sort_order']], on='year_id')
cf = cf.merge(years[['year_id', 'year_label', 'sort_order']], on='year_id')
companies = companies.merge(sectors[['sector_id', 'sector_name']], on='sector_id', how='left')

print(f'Companies: {len(companies)}, P&L rows: {len(pl)}, BS rows: {len(bs)}, CF rows: {len(cf)}')
print(f'Sectors: {sectors.sector_name.nunique()}, Years: {years.year_label.nunique()}')

## 1. Revenue Distribution (Histogram + Box Plot)

In [ ]:
latest_pl = pl.sort_values('sort_order').groupby('company_id').last().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(latest_pl['sales'].dropna(), bins=30, color='#6366F1', edgecolor='white')
axes[0].set_title('Revenue Distribution (Latest Year)', fontweight='bold')
axes[0].set_xlabel('Sales (₹ Cr)')
sns.boxplot(x=latest_pl['sales'].dropna(), ax=axes[1], color='#6366F1')
axes[1].set_title('Revenue Box Plot', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Top 10 & Bottom 10 by Revenue

In [ ]:
top10_rev = latest_pl.nlargest(10, 'sales')[['company_id', 'sales']]
bot10_rev = latest_pl.nsmallest(10, 'sales')[['company_id', 'sales']]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(top10_rev['company_id'], top10_rev['sales'], color='#10B981')
axes[0].set_title('Top 10 Companies by Revenue', fontweight='bold')
axes[0].invert_yaxis()
axes[1].barh(bot10_rev['company_id'], bot10_rev['sales'], color='#EF4444')
axes[1].set_title('Bottom 10 Companies by Revenue', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Top 10 & Bottom 10 by Net Profit

In [ ]:
top10_np = latest_pl.nlargest(10, 'net_profit')[['company_id', 'net_profit']]
bot10_np = latest_pl.nsmallest(10, 'net_profit')[['company_id', 'net_profit']]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(top10_np['company_id'], top10_np['net_profit'], color='#22C55E')
axes[0].set_title('Top 10 by Net Profit', fontweight='bold'); axes[0].invert_yaxis()
axes[1].barh(bot10_np['company_id'], bot10_np['net_profit'], color='#F97316')
axes[1].set_title('Bottom 10 by Net Profit', fontweight='bold'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 4. Top 10 & Bottom 10 by ROE

In [ ]:
roe_df = companies[['symbol', 'company_name', 'roe_pct']].dropna(subset=['roe_pct'])
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
t10 = roe_df.nlargest(10, 'roe_pct')
b10 = roe_df.nsmallest(10, 'roe_pct')
axes[0].barh(t10['symbol'], t10['roe_pct'], color='#10B981')
axes[0].set_title('Top 10 by ROE %', fontweight='bold'); axes[0].invert_yaxis()
axes[1].barh(b10['symbol'], b10['roe_pct'], color='#EF4444')
axes[1].set_title('Bottom 10 by ROE %', fontweight='bold'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 5. Top 10 & Bottom 10 by OPM%

In [ ]:
opm_df = latest_pl[['company_id', 'opm_pct']].dropna(subset=['opm_pct'])
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(opm_df.nlargest(10,'opm_pct')['company_id'], opm_df.nlargest(10,'opm_pct')['opm_pct'], color='#6366F1')
axes[0].set_title('Top 10 by OPM%', fontweight='bold'); axes[0].invert_yaxis()
axes[1].barh(opm_df.nsmallest(10,'opm_pct')['company_id'], opm_df.nsmallest(10,'opm_pct')['opm_pct'], color='#F59E0B')
axes[1].set_title('Bottom 10 by OPM%', fontweight='bold'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 6. Top 10 & Bottom 10 by 3Y Growth

In [ ]:
growth_3y = analysis[analysis['period_label'] == '3Y'][['company_id', 'compounded_sales_growth_pct']].dropna()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
t = growth_3y.nlargest(10, 'compounded_sales_growth_pct')
b = growth_3y.nsmallest(10, 'compounded_sales_growth_pct')
axes[0].barh(t['company_id'], t['compounded_sales_growth_pct'], color='#10B981')
axes[0].set_title('Top 10 by 3Y Sales CAGR', fontweight='bold'); axes[0].invert_yaxis()
axes[1].barh(b['company_id'], b['compounded_sales_growth_pct'], color='#EF4444')
axes[1].set_title('Bottom 10 by 3Y Sales CAGR', fontweight='bold'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 7. Correlation Matrix

In [ ]:
# Build a feature matrix per company
latest_bs = bs.sort_values('sort_order').groupby('company_id').last().reset_index()
feat = latest_pl[['company_id','sales','net_profit','opm_pct','eps','net_profit_margin_pct','interest_coverage']].copy()
feat = feat.merge(latest_bs[['company_id','debt_to_equity','equity_ratio','total_assets']], on='company_id', how='left')
feat = feat.merge(companies[['symbol','roe_pct','roce_pct']], left_on='company_id', right_on='symbol', how='left')
feat = feat.merge(ml_scores[['company_id','overall_score']], on='company_id', how='left')

corr_cols = ['sales','net_profit','opm_pct','eps','debt_to_equity','roe_pct','roce_pct','overall_score','interest_coverage']
corr = feat[corr_cols].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, square=True)
plt.title('Correlation Matrix — Key Financial Metrics', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

## 8. Sector-wise Aggregations (Mean & Median)

In [ ]:
sector_feat = feat.merge(companies[['symbol','sector_name']], left_on='company_id', right_on='symbol', how='left', suffixes=('','_dup'))
sector_agg = sector_feat.groupby('sector_name').agg(
    mean_sales=('sales','mean'), median_sales=('sales','median'),
    mean_opm=('opm_pct','mean'), median_opm=('opm_pct','median'),
    mean_roe=('roe_pct','mean'), mean_de=('debt_to_equity','mean'),
    mean_score=('overall_score','mean'), count=('company_id','count')
).round(2).sort_values('mean_score', ascending=False)
sector_agg

## 9. Null Value Heatmap

In [ ]:
null_data = pd.DataFrame({
    'fact_profit_loss': pl.isnull().sum(),
    'fact_balance_sheet': bs.reindex(columns=pl.columns).isnull().sum(),
}).dropna(how='all').head(15)
# Simpler: show nulls per table
tables = {'P&L': pl, 'Balance Sheet': bs, 'Cash Flow': cf}
null_summary = pd.DataFrame({name: df.isnull().sum() for name, df in tables.items()})
plt.figure(figsize=(14, 8))
sns.heatmap(null_summary.T, cmap='YlOrRd', annot=True, fmt='g')
plt.title('Null Value Counts per Column per Table', fontweight='bold')
plt.tight_layout(); plt.show()

## 10. Year Coverage per Company

In [ ]:
year_cov = pl.groupby('company_id')['year_id'].nunique().sort_values(ascending=True)
plt.figure(figsize=(16, 10))
plt.barh(year_cov.index, year_cov.values, color='#6366F1', height=0.7)
plt.title('Year Coverage per Company (P&L)', fontweight='bold')
plt.xlabel('Number of Years with Data')
plt.tight_layout(); plt.show()

## 11. Sales Growth Rate Distribution

In [ ]:
if len(growth_3y) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(growth_3y['compounded_sales_growth_pct'].dropna(), bins=25, color='#3B82F6', edgecolor='white')
    axes[0].set_title('Distribution of 3Y Sales CAGR', fontweight='bold')
    axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.7)
    from scipy import stats
    vals = growth_3y['compounded_sales_growth_pct'].dropna()
    stat, p_val = stats.normaltest(vals) if len(vals) > 8 else (0, 1)
    axes[1].set_title(f'Q-Q Plot (normality p={p_val:.4f})', fontweight='bold')
    stats.probplot(vals, plot=axes[1])
    plt.tight_layout(); plt.show()

## 12. Outlier Analysis — D/E Ratio

In [ ]:
de_vals = latest_bs[['company_id','debt_to_equity']].dropna()
mean_de = de_vals['debt_to_equity'].mean()
std_de = de_vals['debt_to_equity'].std()
de_vals['z_score'] = (de_vals['debt_to_equity'] - mean_de) / std_de
outliers = de_vals[de_vals['z_score'].abs() > 2]
print(f'D/E Outliers (|z| > 2): {len(outliers)} companies')
print(outliers[['company_id','debt_to_equity','z_score']].to_string())

plt.figure(figsize=(14, 5))
plt.scatter(de_vals['company_id'], de_vals['debt_to_equity'], c='#6366F1', s=50)
plt.scatter(outliers['company_id'], outliers['debt_to_equity'], c='red', s=100, label='Outlier')
plt.axhline(y=mean_de + 2*std_de, color='red', linestyle='--', alpha=0.5)
plt.xticks(rotation=90, fontsize=7)
plt.title('D/E Ratio — Outlier Detection', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

## 13. Outlier Analysis — OPM%

In [ ]:
opm_vals = latest_pl[['company_id','opm_pct']].dropna()
mean_o = opm_vals['opm_pct'].mean()
std_o = opm_vals['opm_pct'].std()
opm_vals['z_score'] = (opm_vals['opm_pct'] - mean_o) / std_o
opm_out = opm_vals[opm_vals['z_score'].abs() > 2]
print(f'OPM% Outliers: {len(opm_out)} companies')
plt.figure(figsize=(12, 5))
sns.boxplot(x=opm_vals['opm_pct'], color='#F59E0B')
plt.title('OPM% Distribution with Outliers', fontweight='bold')
plt.show()

## 14. Sector Distribution (Donut Chart)

In [ ]:
sector_counts = companies['sector_name'].value_counts()
fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(sector_counts, labels=sector_counts.index,
    autopct='%1.1f%%', pctdistance=0.85, startangle=90,
    colors=sns.color_palette('husl', len(sector_counts)))
centre_circle = plt.Circle((0,0), 0.55, fc='white')
ax.add_artist(centre_circle)
ax.set_title('Sector Distribution of Nifty 100 Companies', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

## 15. Health Score Distribution

In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(ml_scores['overall_score'], bins=20, color='#10B981', edgecolor='white')
plt.axvline(x=85, color='green', linestyle='--', label='Excellent threshold')
plt.axvline(x=35, color='red', linestyle='--', label='Weak threshold')
plt.title('ML Health Score Distribution', fontweight='bold', fontsize=14)
plt.xlabel('Overall Score'); plt.ylabel('Count'); plt.legend()
plt.tight_layout(); plt.show()

## 16. Total Nifty 100 Revenue Over Time

In [ ]:
rev_by_year = pl.groupby(['year_label','sort_order'])['sales'].sum().reset_index().sort_values('sort_order')
plt.figure(figsize=(14, 5))
plt.plot(rev_by_year['year_label'], rev_by_year['sales'], marker='o', color='#3B82F6', linewidth=2)
plt.fill_between(rev_by_year['year_label'], rev_by_year['sales'], alpha=0.15, color='#3B82F6')
plt.xticks(rotation=45, fontsize=8)
plt.title('Total Nifty 100 Revenue Over Time', fontweight='bold', fontsize=14)
plt.ylabel('Total Sales (₹ Cr)'); plt.tight_layout(); plt.show()

## 17. Net Profit Trend

In [ ]:
np_by_year = pl.groupby(['year_label','sort_order'])['net_profit'].sum().reset_index().sort_values('sort_order')
plt.figure(figsize=(14, 5))
plt.fill_between(np_by_year['year_label'], np_by_year['net_profit'], alpha=0.3, color='#10B981')
plt.plot(np_by_year['year_label'], np_by_year['net_profit'], color='#10B981', linewidth=2)
plt.xticks(rotation=45, fontsize=8)
plt.title('Total Net Profit Trend', fontweight='bold'); plt.tight_layout(); plt.show()

## 18. Sector Revenue Comparison (Stacked Bar)

In [ ]:
pl_sector = pl.merge(companies[['symbol','sector_name']], left_on='company_id', right_on='symbol', how='left')
# Take last 5 year-labels by sort_order
top_years = years.sort_values('sort_order', ascending=False).head(6)['year_label'].tolist()
pl_5y = pl_sector[pl_sector['year_label'].isin(top_years)]
pivot = pl_5y.pivot_table(index='year_label', columns='sector_name', values='sales', aggfunc='sum').fillna(0)
pivot.plot(kind='bar', stacked=True, figsize=(16, 7), colormap='tab20')
plt.title('Sector Revenue Comparison (Recent Years)', fontweight='bold')
plt.ylabel('Sales (₹ Cr)'); plt.legend(bbox_to_anchor=(1.05, 1), fontsize=8)
plt.tight_layout(); plt.show()

## 19. D/E by Sector (Box Plot)

In [ ]:
bs_sector = latest_bs.merge(companies[['symbol','sector_name']], left_on='company_id', right_on='symbol')
plt.figure(figsize=(16, 6))
order = bs_sector.groupby('sector_name')['debt_to_equity'].median().sort_values().index
sns.boxplot(data=bs_sector, x='sector_name', y='debt_to_equity', order=order, palette='coolwarm')
plt.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='D/E = 1')
plt.xticks(rotation=45); plt.title('Debt-to-Equity by Sector', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

## 20. OPM% Heatmap by Sector × Year

In [ ]:
opm_sector = pl_sector.pivot_table(index='sector_name', columns='year_label', values='opm_pct', aggfunc='mean')
# Keep last 8 years by sort order
recent_labels = years.sort_values('sort_order', ascending=False).head(8)['year_label'].tolist()
opm_recent = opm_sector[[c for c in recent_labels if c in opm_sector.columns]]
plt.figure(figsize=(14, 9))
sns.heatmap(opm_recent, annot=True, fmt='.1f', cmap='RdYlGn', center=15)
plt.title('Average OPM% by Sector × Year', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
conn.close()
print('\n✅ EDA complete — 20 visualizations generated.')